# 08. GRRF-inspired regularized feature selection + SVM

Šis etapas **nekeičia** 01–07 notebook'ų ir neperrašo senų CSV. Tikslas — pirmas korektiškas palyginimas:

- **A)** SVM su visais 561 požymiais (tas pats `LinearSVC`, kaip 3 etape);
- **B)** GRRF principu paremta atranka **tik TRAIN** + tas pats SVM.

Metodas vadinamas **GRRF-inspired regularized feature selection**, nes tai nėra CRAN `RRF` / R realizacija. Paprastas `RandomForestClassifier.feature_importances_` ar `SelectFromModel` čia **nėra** GRRF.

Šaltinis: Thakur ir Biswas (2022), PMC9103613, lygtys (6)–(8).

## Ką darome ir ko nedarome

- Naudojamas jau išsaugotas subject-disjoint skaidymas.
- TEST nenaudojamas importance, atrankai, požymių skaičiui, λ/γ ar SVM `C` parinkti.
- SVM hiperparametrai fiksuoti: `C=0.01` (parinkta 3 etape tik TRAIN).
- GRRF parametrai šiame pirmajame eksperimente **neieškomi TEST**. `γ=1` imamas iš straipsnio (maksimali bauda). `λ` prie `γ=1` į `α_k` neįeina — tai pažymėta žemiau. `n_tree=100` ir `mtry=√p` fiksuoti kaip RF įprasta, be TEST tinklelio.
- Rezultatų „gerinti“ papildomais tinkleliais šiame etape nebandome.

In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

EXPECTED_SVM_F1 = 0.8971298648661375
EXPECTED_SVM_BA = 0.8936657613403356
EXPECTED_SVM_ERR = 347

print(f"RANDOM_STATE = {RANDOM_STATE}")

RANDOM_STATE = 42


## 1. Subject-disjoint TRAIN/TEST

Naujo skaidymo nekuriame. Užkrauname `results/subject_disjoint_split.npz`.

In [2]:
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
project_root = None
for cand in candidates:
    if (cand / "results").exists() and (cand / "notebooks").exists():
        project_root = cand
        break
if project_root is None:
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

results_dir = project_root / "results"
split_path = results_dir / "subject_disjoint_split.npz"
if not split_path.exists():
    raise FileNotFoundError(
        f"Nerastas {split_path}. Pirmiausia paleiskite 01_data_preparation-duomenu_paruosimas.ipynb."
    )

split = np.load(split_path, allow_pickle=True)
X_train = np.asarray(split["X_train_final"], dtype=np.float64)
X_test = np.asarray(split["X_test_final"], dtype=np.float64)
y_train = np.asarray(split["y_train_final"], dtype=np.int64)
y_test = np.asarray(split["y_test_final"], dtype=np.int64)
subjects_train = np.asarray(split["subjects_train_final"], dtype=np.int64)
subjects_test = np.asarray(split["subjects_test_final"], dtype=np.int64)
feature_names = np.array([str(n) for n in split["feature_names"]])
activity_ids = np.asarray(split["activity_ids"], dtype=np.int64)
activity_names = np.array([str(n) for n in split["activity_names"]])
saved_rs = int(split["random_state"][0])

train_ids = sorted(set(subjects_train.tolist()))
test_ids = sorted(set(subjects_test.tolist()))
overlap = sorted(set(train_ids) & set(test_ids))

print(f"Skaidymas: {split_path}")
print(f"Išsaugotas RANDOM_STATE: {saved_rs}")
print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  y_test: {y_test.shape}")
print(f"TRAIN subject ID ({len(train_ids)}): {train_ids}")
print(f"TEST subject ID  ({len(test_ids)}): {test_ids}")
print(f"TRAIN ∩ TEST subject: {overlap}")

assert overlap == [], f"Data leakage: subject sankirta {overlap}"
assert X_train.shape == (7144, 561)
assert X_test.shape == (3155, 561)
assert len(train_ids) == 21 and len(test_ids) == 9
assert len(feature_names) == 561
print("Assertion: sankirta tuščia, dydžiai 7144/21 ir 3155/9.")

Skaidymas: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\subject_disjoint_split.npz
Išsaugotas RANDOM_STATE: 42
X_train: (7144, 561)  y_train: (7144,)
X_test:  (3155, 561)  y_test: (3155,)
TRAIN subject ID (21): [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15, 17, 19, 20, 21, 22, 23, 26, 27, 30]
TEST subject ID  (9): [9, 10, 13, 16, 18, 24, 25, 28, 29]
TRAIN ∩ TEST subject: []
Assertion: sankirta tuščia, dydžiai 7144/21 ir 3155/9.


## 2. GRRF-inspired: kas realizuojama ir kas skiriasi

Straipsnio esmė:

- (5) RF Gini importance visame TRAIN.
- (6) RRF: jei požymis dar ne F, gain × λ.
- (7)–(8) GRRF: jei požymis dar ne F, gain × α_k, kur  
  `α_k = (1−γ)λ + γ (Impscore_k / Impscore_max)`.
- Atranka: požymiai, kuriuos regularizuotas miškas panaudojo skėlimams (kaip `RRF$feaSet`), **ne** top-K pagal importance.

Šiame Python kode:

1. (5) skaičiuoja `sklearn` RF **tik TRAIN** — tai tik importance, ne atranka.
2. (8) normalizuoja max ir skaičiuoja `α_k` su `γ=1` (straipsnis: maksimali bauda).
3. (7) kiekviename medyje, kiekviename mazge: naujo požymio Gini gain dauginamas iš `α_k`; jau F esančio — ne.
4. F = požymiai, jau panaudoti **tame pačiame medyje** ankstesniuose mazguose (depth-first). Originalus Fortran `RRF` mazgų eilės gali skirtis.
5. **Nenaudojame** R/`rpy2`. Tai **nėra** identiškas CRAN `RRF`.

Kai `γ=1`, (8) tampa `α_k = Impscore_k / Impscore_max`, todėl `λ` **nedalyvauja**. Straipsnis vis tiek mini `λ` tinklelyje — čia to neimituojame tyliai, o išvedame perspėjimą.

In [3]:
def gini_parent(counts):
    n = counts.sum()
    if n <= 0:
        return 0.0
    return 1.0 - float(np.dot(counts, counts)) / float(n * n)


def best_threshold_and_gain(x_col, y_node, n_classes, min_leaf):
    """Gini gain ir slenkstis vienam požymiui (straipsnio (4) lygties skaitmeninis analogas)."""
    n = x_col.shape[0]
    if n < 2 * min_leaf:
        return -1.0, np.nan
    order = np.argsort(x_col, kind="mergesort")
    x_s = x_col[order]
    y_s = y_node[order]
    if x_s[0] == x_s[-1]:
        return -1.0, np.nan
    oh = np.zeros((n, n_classes), dtype=np.float64)
    oh[np.arange(n), y_s] = 1.0
    left = np.cumsum(oh, axis=0)
    total = left[-1]
    g_parent = gini_parent(total)
    i = np.arange(min_leaf, n - min_leaf + 1)
    valid = x_s[i] != x_s[i - 1]
    if not np.any(valid):
        return -1.0, np.nan
    left_c = left[i - 1]
    left_n = i.astype(np.float64)
    right_c = total - left_c
    right_n = float(n) - left_n
    g_left = 1.0 - np.sum(left_c * left_c, axis=1) / (left_n * left_n)
    g_right = 1.0 - np.sum(right_c * right_c, axis=1) / (right_n * right_n)
    gain = g_parent - (left_n / float(n)) * g_left - (right_n / float(n)) * g_right
    gain = np.where(valid, gain, -np.inf)
    j = int(np.argmax(gain))
    thr = 0.5 * (x_s[i[j]] + x_s[i[j] - 1])
    return float(gain[j]), float(thr)


def grow_regularized_tree(
    X, y, rng, alpha, max_features, min_samples_split, min_samples_leaf, max_depth
):
    """Vienas medis su (6)–(7) regularizacija. F pildomas medžio augimo tvarka."""
    n_samples, n_features = X.shape
    n_classes = int(y.max()) + 1
    used = set()  # F: jau naudoti šio medžio ankstesniuose mazguose
    used_in_tree = set()

    def majority(idx):
        counts = np.bincount(y[idx], minlength=n_classes)
        return int(np.argmax(counts))

    def recurse(idx, depth):
        y_n = y[idx]
        if (
            idx.size < min_samples_split
            or np.unique(y_n).size == 1
            or (max_depth is not None and depth >= max_depth)
        ):
            return {"leaf": True, "pred": majority(idx)}

        mtry = min(max_features, n_features)
        cand = rng.choice(n_features, size=mtry, replace=False)
        best_gain = -1.0
        best_k = None
        best_thr = None
        x_node = X[idx]

        for k in cand:
            raw_gain, thr = best_threshold_and_gain(
                x_node[:, k], y_n, n_classes, min_samples_leaf
            )
            if raw_gain < 0 or not np.isfinite(thr):
                continue
            # (6)–(7): k jau F → baudos nėra; k ∉ F → gain × α_k (arba λ paprastame RRF).
            if k in used:
                gain = raw_gain
            else:
                gain = raw_gain * float(alpha[k])
            if gain > best_gain:
                best_gain = gain
                best_k = int(k)
                best_thr = thr

        if best_k is None or best_gain <= 0:
            return {"leaf": True, "pred": majority(idx)}

        used.add(best_k)
        used_in_tree.add(best_k)
        left_mask = x_node[:, best_k] <= best_thr
        left_idx = idx[left_mask]
        right_idx = idx[~left_mask]
        if left_idx.size == 0 or right_idx.size == 0:
            return {"leaf": True, "pred": majority(idx)}
        return {
            "leaf": False,
            "feature": best_k,
            "threshold": best_thr,
            "left": recurse(left_idx, depth + 1),
            "right": recurse(right_idx, depth + 1),
        }

    root_idx = np.arange(n_samples, dtype=np.int64)
    root = recurse(root_idx, 0)
    return used_in_tree, root


def grrf_inspired_select(
    X_train_only,
    y_train_only,
    gamma=1.0,
    lambda_penalty=1.0,
    n_estimators=100,
    random_state=42,
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=None,
):
    """GRRF-inspired regularized feature selection. TEST masyvo čia nėra."""
    warnings.warn(
        "GRRF-inspired regularized feature selection: tai nėra CRAN RRF / Thakur bit-level kopija. "
        "Medžiai auginami Python CART su (6)–(8) baudomis, ne Fortran randomForest.",
        UserWarning,
        stacklevel=2,
    )
    if abs(gamma - 1.0) < 1e-12:
        warnings.warn(
            "γ=1 (straipsnis: maksimali bauda). Tada (8) lygtis α_k = Imp_k/Imp_max, "
            "todėl λ į α_k neįeina ir čia nenaudojamas kaip atskiras tinklelio parametras.",
            UserWarning,
            stacklevel=2,
        )

    n_features = X_train_only.shape[1]
    max_features = max(1, int(np.sqrt(n_features)))

    # --- (5) lygtis: RF importance tik TRAIN, visiems medžiams agreguota ---
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_features="sqrt",
        random_state=random_state,
        n_jobs=-1,
    )
    rf.fit(X_train_only, y_train_only)
    imp = rf.feature_importances_.astype(np.float64)
    imp_max = float(imp.max()) if imp.size else 0.0
    if imp_max <= 0:
        raise RuntimeError("RF importance visur 0 — GRRF-inspired α_k apskaičiuoti negalima.")
    imp_norm = imp / imp_max  # Impscore_k / Impscore_max ∈ [0,1], (8) lygtis

    # --- (8) lygtis: α_k = (1-γ)λ + γ * Imp_norm ---
    alpha = (1.0 - gamma) * lambda_penalty + gamma * imp_norm
    alpha = np.clip(alpha, 0.0, 1.0)

    classes, y_enc = np.unique(y_train_only, return_inverse=True)
    rng = np.random.RandomState(random_state)
    selected = set()
    t0 = time.perf_counter()

    for t in range(n_estimators):
        boot = rng.randint(0, X_train_only.shape[0], size=X_train_only.shape[0])
        Xb = X_train_only[boot]
        yb = y_enc[boot]
        used_t, _ = grow_regularized_tree(
            Xb,
            yb,
            rng,
            alpha,
            max_features=max_features,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_depth=max_depth,
        )
        selected |= used_t
        if (t + 1) % 10 == 0 or t == 0:
            print(
                f"  GRRF-inspired medis {t + 1}/{n_estimators}; "
                f"unikalūs požymiai kol kas {len(selected)}; "
                f"{time.perf_counter() - t0:.1f}s"
            )

    selected_idx = np.array(sorted(selected), dtype=np.int64)
    return {
        "selected_idx": selected_idx,
        "importance": imp,
        "importance_normalized": imp_norm,
        "alpha": alpha,
        "gamma": gamma,
        "lambda_penalty": lambda_penalty,
        "n_estimators": n_estimators,
        "max_features": max_features,
        "classes": classes,
        "rf_for_importance_only": rf,
    }

## 3. Atranka tik TRAIN

`X_test` į `grrf_inspired_select` neperduodamas. Parametrai fiksuoti (`γ=1`, `n_tree=100`), todėl GroupKFold tinklelio šiame pirmajame paleidime nėra — TEST vis tiek neliestas. Jei kitame etape reikėtų rinkti `n_tree`/`γ`, tai darytina tik TRAIN `GroupKFold`.

In [4]:
print("GRRF-inspired atranka: fit tik X_train / y_train.")
print("X_test.shape (neturi būti naudojamas čia):", X_test.shape)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    grrf = grrf_inspired_select(
        X_train,
        y_train,
        gamma=1.0,
        lambda_penalty=1.0,
        n_estimators=100,
        random_state=RANDOM_STATE,
    )
    for w in caught:
        print("PERSPĖJIMAS:", str(w.message))

sel = grrf["selected_idx"]
n_sel = int(sel.size)
n_removed = 561 - n_sel
pct_removed = 100.0 * n_removed / 561.0
sel_names = feature_names[sel]

print()
print("γ =", grrf["gamma"], " (straipsnis: maksimali bauda)")
print("λ pateiktas funkcijai, bet prie γ=1 į α_k neįeina:", grrf["lambda_penalty"])
print("n_estimators =", grrf["n_estimators"], " mtry =", grrf["max_features"])
print(f"Atrinkta požymių: {n_sel} / 561")
print(f"Pašalinta: {n_removed} ({pct_removed:.2f} %)")
print("Indeksai:", sel.tolist())
print("Pavadinimai:")
for i, name in zip(sel, sel_names):
    print(f"  {int(i):3d}  {name}")

print("\nα_k santrauka (tik TRAIN importance):")
print("  min/mean/max α_k:", float(grrf["alpha"].min()), float(grrf["alpha"].mean()), float(grrf["alpha"].max()))
print("  požymių su α_k=0:", int(np.sum(grrf["alpha"] == 0)))

GRRF-inspired atranka: fit tik X_train / y_train.
X_test.shape (neturi būti naudojamas čia): (3155, 561)


  GRRF-inspired medis 1/100; unikalūs požymiai kol kas 70; 0.6s


  GRRF-inspired medis 10/100; unikalūs požymiai kol kas 159; 5.7s


  GRRF-inspired medis 20/100; unikalūs požymiai kol kas 180; 11.4s


  GRRF-inspired medis 30/100; unikalūs požymiai kol kas 188; 17.4s


...

  GRRF-inspired medis 60/100; unikalūs požymiai kol kas 224; 34.4s


  GRRF-inspired medis 70/100; unikalūs požymiai kol kas 235; 40.4s


  GRRF-inspired medis 80/100; unikalūs požymiai kol kas 244; 46.3s


  GRRF-inspired medis 90/100; unikalūs požymiai kol kas 247; 52.1s


  GRRF-inspired medis 100/100; unikalūs požymiai kol kas 254; 58.0s
PERSPĖJIMAS: GRRF-inspired regularized feature selection: tai nėra CRAN RRF / Thakur bit-level kopija. Medžiai auginami Python CART su (6)–(8) baudomis, ne Fortran randomForest.
PERSPĖJIMAS: γ=1 (straipsnis: maksimali bauda). Tada (8) lygtis α_k = Imp_k/Imp_max, todėl λ į α_k neįeina ir čia nenaudojamas kaip atskiras tinklelio parametras.

γ = 1.0  (straipsnis: maksimali bauda)
λ pateiktas funkcijai, bet prie γ=1 į α_k neįeina: 1.0
n_estimators = 100  mtry = 23
Atrinkta požymių: 254 / 561
Pašalinta: 307 (54.72 %)
Indeksai: [3, 6, 7, 9, 12, 15, 16, 17, 19, 22, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 60, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 83, 85, 86, 87, 88, 89, 91, 92, 95, 96, 98, 99, 100, 102, 103, 107, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 128, 129, 132, 134, 136, 138, 139, 141, 142, 144, 145, 149, 150, 158, 159, 163, 164, 166, 168, 1

## 4. A vs B: tas pats SVM, skiriasi tik požymiai

Abu: `StandardScaler` + `LinearSVC(C=0.01, max_iter=10000, dual="auto")`, `fit` tik TRAIN. TEST — tik `predict` po `fit`.

In [5]:
def make_svm():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LinearSVC(
                    C=0.01,
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    dual="auto",
                ),
            ),
        ]
    )


def eval_block(name, y_true, y_pred):
    return {
        "Model": name,
        "N features": None,
        "Macro-F1": float(f1_score(y_true, y_pred, average="macro")),
        "Balanced Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "TEST Errors": int(np.sum(y_pred != y_true)),
    }


pipe_a = make_svm()
pipe_a.fit(X_train, y_train)
pred_a = pipe_a.predict(X_test)
row_a = eval_block("SVM (all 561 features)", y_test, pred_a)
row_a["N features"] = 561

print("A) Baseline SVM TEST")
print(
    f"  Macro-F1={row_a['Macro-F1']:.12f}  BA={row_a['Balanced Accuracy']:.12f}  "
    f"Acc={row_a['Accuracy']:.6f}  errors={row_a['TEST Errors']}"
)

f1_ok = np.isclose(row_a["Macro-F1"], EXPECTED_SVM_F1, rtol=0.0, atol=1e-10)
ba_ok = np.isclose(row_a["Balanced Accuracy"], EXPECTED_SVM_BA, rtol=0.0, atol=1e-10)
err_ok = row_a["TEST Errors"] == EXPECTED_SVM_ERR
print(f"  Sutampa su 3 etapo SVM? Macro-F1={f1_ok}, BA={ba_ok}, klaidos={err_ok}")

if not (f1_ok and ba_ok and err_ok):
    raise RuntimeError(
        "A variantas NESUTAMPA su ankstesniu SVM. B rezultatų neinterpretuojame. "
        f"Gauta F1={row_a['Macro-F1']!r} BA={row_a['Balanced Accuracy']!r} "
        f"err={row_a['TEST Errors']}; tikėtasi F1={EXPECTED_SVM_F1} "
        f"BA={EXPECTED_SVM_BA} err={EXPECTED_SVM_ERR}."
    )
print("A reprodukuotas. Tęsiame B.")

A) Baseline SVM TEST
  Macro-F1=0.897129864866  BA=0.893665761340  Acc=0.890016  errors=347
  Sutampa su 3 etapo SVM? Macro-F1=True, BA=True, klaidos=True
A reprodukuotas. Tęsiame B.


In [6]:
X_train_b = X_train[:, sel]
X_test_b = X_test[:, sel]

pipe_b = make_svm()
pipe_b.fit(X_train_b, y_train)
pred_b = pipe_b.predict(X_test_b)
row_b = eval_block("GRRF-inspired + SVM", y_test, pred_b)
row_b["N features"] = n_sel

print("B) GRRF-inspired + SVM TEST")
print(
    f"  Macro-F1={row_b['Macro-F1']:.12f}  BA={row_b['Balanced Accuracy']:.12f}  "
    f"Acc={row_b['Accuracy']:.6f}  errors={row_b['TEST Errors']}"
)

labels = activity_ids.tolist()
cm_a = confusion_matrix(y_test, pred_a, labels=labels)
cm_b = confusion_matrix(y_test, pred_b, labels=labels)
id_to_name = {int(i): str(n) for i, n in zip(activity_ids, activity_names)}
col_names = [id_to_name[int(i)] for i in labels]

print("\nA painiavos matrica (eilutė=tikra, stulpelis=prognozė)")
print(pd.DataFrame(cm_a, index=col_names, columns=col_names).to_string())
print("\nB painiavos matrica")
print(pd.DataFrame(cm_b, index=col_names, columns=col_names).to_string())

cmp = pd.DataFrame([row_a, row_b])
cmp["Macro-F1 Δ vs A"] = cmp["Macro-F1"] - row_a["Macro-F1"]
cmp["BA Δ vs A"] = cmp["Balanced Accuracy"] - row_a["Balanced Accuracy"]
cmp["Errors Δ vs A"] = cmp["TEST Errors"] - row_a["TEST Errors"]
print("\nA vs B")
print(cmp.to_string(index=False))

B) GRRF-inspired + SVM TEST
  Macro-F1=0.881137370673  BA=0.876256430825  Acc=0.873851  errors=398

A painiavos matrica (eilutė=tikra, stulpelis=prognozė)
                    WALKING  WALKING_UPSTAIRS  WALKING_DOWNSTAIRS  SITTING  STANDING  LAYING
WALKING                 505                 2                   0        1         0       0
WALKING_UPSTAIRS         10               414                   1       59         0       0
WALKING_DOWNSTAIRS        3                18                 414        1         0       0
SITTING                   0                 2                   0      472        70       0
STANDING                  0                 0                   0       97       487       0
LAYING                    0                 0                   0        0        83     516

B painiavos matrica
                    WALKING  WALKING_UPSTAIRS  WALKING_DOWNSTAIRS  SITTING  STANDING  LAYING
WALKING                 501                 2                   0        5      

## 5. Išsaugojimas (nauji failai, senų neperrašome)

In [7]:
feat_path = results_dir / "grrf_inspired_selected_features.csv"
cmp_path = results_dir / "grrf_inspired_vs_svm.csv"
cm_path = results_dir / "grrf_inspired_svm_confusion_matrix.csv"

feat_df = pd.DataFrame(
    {
        "feature_index": sel,
        "feature_name": sel_names,
        "rf_importance_train_only": grrf["importance"][sel],
        "importance_normalized_eq8": grrf["importance_normalized"][sel],
        "alpha_k_eq8": grrf["alpha"][sel],
    }
)
feat_df.to_csv(feat_path, index=False)

cmp.to_csv(cmp_path, index=False)

cm_df = pd.DataFrame(cm_b, index=col_names, columns=col_names)
cm_df.insert(0, "true_activity", col_names)
cm_df.to_csv(cm_path, index=False)

print("Išsaugota:")
print(" ", feat_path)
print(" ", cmp_path)
print(" ", cm_path)
print("Seni failai (final_model_comparison.csv ir kt.) neperrašyti.")

Išsaugota:
  C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\grrf_inspired_selected_features.csv
  C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\grrf_inspired_vs_svm.csv
  C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\grrf_inspired_svm_confusion_matrix.csv
Seni failai (final_model_comparison.csv ir kt.) neperrašyti.


## 6. Data leakage auditas

Konkretūs įrodymai iš šio notebook vykdymo, ne iš ketinimų.

In [8]:
print("=== DATA LEAKAGE AUDITAS ===")
print("1) TRAIN subject ID:", train_ids)
print("   TEST subject ID:", test_ids)
print("   sankirta:", overlap, " tuščia=", overlap == [])
print()
print("2) RF importance fitintas ant X_train, y_train; n_features_in_=",
      grrf["rf_for_importance_only"].n_features_in_)
print("   RF n_samples_:", getattr(grrf["rf_for_importance_only"], "n_samples_", X_train.shape[0]))
print("   X_train eilučių:", X_train.shape[0], "(turi būti 7144, ne 3155 ir ne 10299)")
print()
print("3) Regularizuoti medžiai auginti tik iš X_train bootstrap (žr. grrf_inspired_select).")
print("   Atrinktų indeksų skaičius:", n_sel, "— kaukė sudaryta prieš B SVM fit.")
print()
print("4) StandardScaler A: n_features_in_=", pipe_a.named_steps["scaler"].n_features_in_,
      " mean_len=", len(pipe_a.named_steps["scaler"].mean_))
print("   StandardScaler B: n_features_in_=", pipe_b.named_steps["scaler"].n_features_in_,
      " (turi būti", n_sel, ")")
print("   Scaler mean skaičiuotas Pipeline.fit(TRAIN) metu, ne TEST.")
print()
print("5) SVM A classes_:", pipe_a.named_steps["clf"].classes_)
print("   SVM B classes_:", pipe_b.named_steps["clf"].classes_)
print("   LinearSVC C visur 0.01, neperrinktas pagal TEST.")
print()
print("6) TEST naudotas tik po A ir B fit, funkcijose predict / metrikos / confusion_matrix.")
print("   TEST nebuvo paduotas į RandomForestClassifier.fit, grrf_inspired_select, GroupKFold ar C paiešką.")
print()
print("7) Hipotezė (šiame etape): TRAIN-only GRRF-inspired duoda mažesnį požymių rinkinį ")
print("   neprarandant Macro-F1 / BA subject-disjoint TEST, palyginti su 561 LinearSVC.")
d_f1 = row_b["Macro-F1"] - row_a["Macro-F1"]
d_ba = row_b["Balanced Accuracy"] - row_a["Balanced Accuracy"]
d_err = row_b["TEST Errors"] - row_a["TEST Errors"]
print(f"   Δ Macro-F1={d_f1:+.6f}  Δ BA={d_ba:+.6f}  Δ klaidos={d_err:+d}")
if n_sel < 561 and abs(d_f1) < 0.01 and abs(d_ba) < 0.01:
    print("   Interpretacija: kompaktiškesnis rinkinys, metrikos labai artimos — hipotezė šio paleidimo ribose palaikoma.")
elif n_sel < 561 and d_f1 < -0.01:
    print("   Interpretacija: požymių sumažėjo, bet Macro-F1 krito — hipotezė šio paleidimo ribose nepasitvirtino.")
elif n_sel < 561 and d_f1 > 0.01:
    print("   Interpretacija: mažiau požymių ir aukštesnis Macro-F1 šiame TEST; vis tiek ne 99.30 % straipsnio protocol.")
else:
    print("   Interpretacija: žr. skaičius; nelyginti su straipsnio 99.30 % (kitas TEST ir kitas SVM).")

=== DATA LEAKAGE AUDITAS ===
1) TRAIN subject ID: [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15, 17, 19, 20, 21, 22, 23, 26, 27, 30]
   TEST subject ID: [9, 10, 13, 16, 18, 24, 25, 28, 29]
   sankirta: []  tuščia= True

2) RF importance fitintas ant X_train, y_train; n_features_in_= 561
   RF n_samples_: 7144
   X_train eilučių: 7144 (turi būti 7144, ne 3155 ir ne 10299)

3) Regularizuoti medžiai auginti tik iš X_train bootstrap (žr. grrf_inspired_select).
   Atrinktų indeksų skaičius: 254 — kaukė sudaryta prieš B SVM fit.

4) StandardScaler A: n_features_in_= 561  mean_len= 561
   StandardScaler B: n_features_in_= 254  (turi būti 254 )
   Scaler mean skaičiuotas Pipeline.fit(TRAIN) metu, ne TEST.

5) SVM A classes_: [1 2 3 4 5 6]
   SVM B classes_: [1 2 3 4 5 6]
   LinearSVC C visur 0.01, neperrinktas pagal TEST.

6) TEST naudotas tik po A ir B fit, funkcijose predict / metrikos / confusion_matrix.
   TEST nebuvo paduotas į RandomForestClassifier.fit, grrf_inspired_select, GroupKFold ar C p